In [1]:
import pandas as pd
import numpy as np

import json

import re

import warnings
warnings.filterwarnings("ignore")

In [21]:
import nltk
nltk.download()

showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml


True

In [22]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from rake_nltk import Rake

In [23]:
FILEPATH="../data/cleaned/"

df = pd.read_csv(FILEPATH+"movie_data_cleaned.csv")

In [24]:
df.head()

,title,genres,keywords,overview
0,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","Captain Barbossa, long believed to be dead, ha..."
2,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",Following the death of District Attorney Harve...
4,John Carter,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","John Carter is a war-weary, former military ca..."


In [25]:
df.loc[0, 'genres']

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [26]:
json.loads(df.loc[0, 'genres'])

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 878, 'name': 'Science Fiction'}]

# Data Extraction

### Extracting genres and keywords and storing into dataset

In [27]:
def data_extraction(x):
    data_list = []
    x = json.loads(x)
    for i, item in enumerate(x):
        data_list.append(x[i]['name'])
    final_data = ', '.join(data_list)
    return final_data

In [28]:
df.loc[:, 'genres'] = df.loc[:, 'genres'].apply(lambda x: data_extraction(x))
df.loc[:, 'keywords'] = df.loc[:, 'keywords'].apply(lambda x: data_extraction(x))

In [29]:
df.head()

,title,genres,keywords,overview
0,Avatar,"Action, Adventure, Fantasy, Science Fiction","culture clash, future, space war, space colony...","In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Adventure, Fantasy, Action","ocean, drug abuse, exotic island, east india t...","Captain Barbossa, long believed to be dead, ha..."
2,Spectre,"Action, Adventure, Crime","spy, based on novel, secret agent, sequel, mi6...",A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,"Action, Crime, Drama, Thriller","dc comics, crime fighter, terrorist, secret id...",Following the death of District Attorney Harve...
4,John Carter,"Action, Adventure, Science Fiction","based on novel, mars, medallion, space travel,...","John Carter is a war-weary, former military ca..."


In [30]:
df.isnull().sum()

title       0
genres      0
keywords    0
overview    3
dtype: int64

# Checking Available Genres:

In [31]:
x = df.loc[0,'genres']
x.split(', ')

['Action', 'Adventure', 'Fantasy', 'Science Fiction']

In [32]:
genres = set()
for i in df['genres']:
    i = i.split(', ')
    genres.update(i)

In [33]:
genres

{'',
 'Action',
 'Adventure',
 'Animation',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Family',
 'Fantasy',
 'Foreign',
 'History',
 'Horror',
 'Music',
 'Mystery',
 'Romance',
 'Science Fiction',
 'TV Movie',
 'Thriller',
 'War',
 'Western'}

In [34]:
genres.remove('')

In [35]:
genres

{'Action',
 'Adventure',
 'Animation',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Family',
 'Fantasy',
 'Foreign',
 'History',
 'Horror',
 'Music',
 'Mystery',
 'Romance',
 'Science Fiction',
 'TV Movie',
 'Thriller',
 'War',
 'Western'}

In [36]:
df['overview'] = df['overview'].fillna(value='')

In [37]:
df['overview'].isnull().sum() # no null values present

np.int64(0)

In [38]:
description = df.loc[0,'overview']
description

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.'

In [39]:
# customize stopwords according to your requirement
cust_stopwords = set(stopwords.words('english'))

### We are using Rake algorithm for quick text extraction

In [40]:
def text_extraction(x):
    r = Rake(stopwords=cust_stopwords)
    r.extract_keywords_from_text(x)
    keyword_score = r.get_word_degrees()
    res = ' '.join(list(keyword_score.keys()))
    return res

In [41]:
df.loc[:, 'overview'] = df.loc[:, 'overview'].apply(lambda x: text_extraction(x))

In [42]:
description = df.loc[0,'overview']
description

'22nd century paraplegic marine dispatched moon pandora unique mission becomes torn following orders protecting alien civilization'

In [43]:
df.head()

,title,genres,keywords,overview
0,Avatar,"Action, Adventure, Fantasy, Science Fiction","culture clash, future, space war, space colony...",22nd century paraplegic marine dispatched moon...
1,Pirates of the Caribbean: At World's End,"Adventure, Fantasy, Action","ocean, drug abuse, exotic island, east india t...",captain barbossa long believed dead come back ...
2,Spectre,"Action, Adventure, Crime","spy, based on novel, secret agent, sequel, mi6...",cryptic message bond ’ past sends trail uncove...
3,The Dark Knight Rises,"Action, Crime, Drama, Thriller","dc comics, crime fighter, terrorist, secret id...",following death district attorney harvey dent ...
4,John Carter,"Action, Adventure, Science Fiction","based on novel, mars, medallion, space travel,...",john carter war weary former military captain ...


In [44]:
df = df.reset_index(drop=True)

In [45]:
for item in df.iterrows():
    print(item)

(0, title                                                  Avatar
genres            Action, Adventure, Fantasy, Science Fiction
keywords    culture clash, future, space war, space colony...
overview    22nd century paraplegic marine dispatched moon...
Name: 0, dtype: object)
(1, title                Pirates of the Caribbean: At World's End
genres                             Adventure, Fantasy, Action
keywords    ocean, drug abuse, exotic island, east india t...
overview    captain barbossa long believed dead come back ...
Name: 1, dtype: object)
(2, title                                                 Spectre
genres                               Action, Adventure, Crime
keywords    spy, based on novel, secret agent, sequel, mi6...
overview    cryptic message bond ’ past sends trail uncove...
Name: 2, dtype: object)
(3, title                                   The Dark Knight Rises
genres                         Action, Crime, Drama, Thriller
keywords    dc comics, crime fighter, terror

In [46]:
for index, row in df.iterrows():
    print("{}\n\n{}\n".format(index, row))

0

title                                                  Avatar
genres            Action, Adventure, Fantasy, Science Fiction
keywords    culture clash, future, space war, space colony...
overview    22nd century paraplegic marine dispatched moon...
Name: 0, dtype: object

1

title                Pirates of the Caribbean: At World's End
genres                             Adventure, Fantasy, Action
keywords    ocean, drug abuse, exotic island, east india t...
overview    captain barbossa long believed dead come back ...
Name: 1, dtype: object

2

title                                                 Spectre
genres                               Action, Adventure, Crime
keywords    spy, based on novel, secret agent, sequel, mi6...
overview    cryptic message bond ’ past sends trail uncove...
Name: 2, dtype: object

3

title                                   The Dark Knight Rises
genres                         Action, Crime, Drama, Thriller
keywords    dc comics, crime fighter, terrorist,

``` python
df['meta_tags'] = '' # creating an empty column


for index, row in df.iterrows():
    #meta tags
    genre = row['genres'].replace(',', '').lower()
    keyword = row['keywords'].replace(',', '').lower()
    overview = row['overview'].lower()
    overview_filtered = re.sub(pattern="[^a-z0-9 ]+", repl="", string=overview)

    # combine all features in to meta tags - string concatination
    combined_keywords = genre+' '+keyword+' '+overview_filtered

    df.at[index, 'meta_tags'] = combined_keywords
```

### Creating weighted meta data

In [47]:
# clean columns
df['genres_clean'] = df['genres'].str.replace(',', '', regex=False).str.lower()
df['keywords_clean'] = df['keywords'].str.replace(',', '', regex=False).str.lower()

df['overview_clean'] = (
    df['overview']
    .str.lower()
    .str.replace(r'[^a-z0-9 ]+', '', regex=True)
)

# combine into meta_tags
df['meta_tags'] = (
    df['genres_clean'] + ' ' +
    df['keywords_clean'] + ' ' +
    df['overview_clean']
)

In [48]:
df.head()

,title,genres,keywords,overview,genres_clean,keywords_clean,overview_clean,meta_tags
0,Avatar,"Action, Adventure, Fantasy, Science Fiction","culture clash, future, space war, space colony...",22nd century paraplegic marine dispatched moon...,action adventure fantasy science fiction,culture clash future space war space colony so...,22nd century paraplegic marine dispatched moon...,action adventure fantasy science fiction cultu...
1,Pirates of the Caribbean: At World's End,"Adventure, Fantasy, Action","ocean, drug abuse, exotic island, east india t...",captain barbossa long believed dead come back ...,adventure fantasy action,ocean drug abuse exotic island east india trad...,captain barbossa long believed dead come back ...,adventure fantasy action ocean drug abuse exot...
2,Spectre,"Action, Adventure, Crime","spy, based on novel, secret agent, sequel, mi6...",cryptic message bond ’ past sends trail uncove...,action adventure crime,spy based on novel secret agent sequel mi6 bri...,cryptic message bond past sends trail uncover...,action adventure crime spy based on novel secr...
3,The Dark Knight Rises,"Action, Crime, Drama, Thriller","dc comics, crime fighter, terrorist, secret id...",following death district attorney harvey dent ...,action crime drama thriller,dc comics crime fighter terrorist secret ident...,following death district attorney harvey dent ...,action crime drama thriller dc comics crime fi...
4,John Carter,"Action, Adventure, Science Fiction","based on novel, mars, medallion, space travel,...",john carter war weary former military captain ...,action adventure science fiction,based on novel mars medallion space travel pri...,john carter war weary former military captain ...,action adventure science fiction based on nove...


In [49]:
df.loc[0,'meta_tags']

'action adventure fantasy science fiction culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d 22nd century paraplegic marine dispatched moon pandora unique mission becomes torn following orders protecting alien civilization'

In [50]:
df_meta = df[['title', 'meta_tags']]

### We are using lemmatization as lexicon normalization instead of using stemming because meta data is sensitive and needs to be clipped carefully without changing the meaning.

In [51]:
def lemmatizer_word(x):
    lemm_obj = WordNetLemmatizer()
    word_tokens = nltk.word_tokenize(text=x, language='english', preserve_line=False)
    lemm_words = []
    for i in word_tokens:
        lemm_words.append(lemm_obj.lemmatize(i))
    res = ' '.join(lemm_words)
    return res

In [52]:
df_meta.loc[:, 'meta_tags'] = df_meta.loc[:, 'meta_tags'].apply(lambda x: lemmatizer_word(x))

In [53]:
df_meta.loc[0, 'meta_tags']

'action adventure fantasy science fiction culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relation mind and soul 3d 22nd century paraplegic marine dispatched moon pandora unique mission becomes torn following order protecting alien civilization'

### Exporting file

In [54]:
EXPORT_FILEPATH="../data/train/"

df_meta.to_csv(EXPORT_FILEPATH+"meta_data.csv", index=False)